# 02 Handmade Augmentation and Error Analysis

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/logs)
- [`EXP_CMB_004_SVC_Combined_Error_Analysis.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/scripts/EXP_CMB_004_SVC_Combined_Error_Analysis.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/logs)
- [`EXP_CMB_005_SVC_Combined_Targeted_DB_Augmentation.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/scripts/EXP_CMB_005_SVC_Combined_Targeted_DB_Augmentation.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '754b9aeffc53822945324de09096c81ff42c501f'}, '05_model_optimize/SVC/Combined/scripts/EXP_CMB_004_SVC_Combined_Error_Analysis.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '2f3412dfc2234f7c2db570449451253a4cb7f310'}, '05_model_optimize/SVC/Combined/scripts/EXP_CMB_005_SVC_Combined_Targeted_DB_Augmentation.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '7e5396776c1a80de99af564542d035cb8cf0df9d'}}

## EXP_CMB_003_SVC_Combined_Handmade_Augmentation

원본: [05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py)  
Git blob: `754b9aeffc53822945324de09096c81ff42c501f`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/logs)

SVC Combined EXP-CMB-003: full official training, writer-heldout augmentation.

Run from repository root:
 python 05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py
 python 05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py --ratio 50
 python 05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py --ratio 100

CLI 인자 (정적 추출): `--ratio`, `--batch-size`, `--cache-size`, `--overwrite`

In [ ]:
%%archive_source 05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SVC Combined EXP-CMB-003: full official training, writer-heldout augmentation.

Run from repository root:
 python 05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py
 python 05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py --ratio 50
 python 05_model_optimize/SVC/Combined/scripts/EXP_CMB_003_SVC_Combined_Handmade_Augmentation.py --ratio 100
"""
from __future__ import annotations

import argparse
import csv
import gc
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.base import clone
from sklearn.metrics import confusion_matrix

ROOT = Path(__file__).resolve().parents[4]
sys.path[:0] = [
    str(ROOT / '05_model_optimize' / 'SVC' / 'Combined' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'KNN' / 'Combined' / 'scripts'),
    str(ROOT / '03_experiments' / 'model_optimization'),
]
from EXP_001_SVC_Normal_Baseline import WRITERS, baseline, load_writer, report
from EXP_005_SVC_Normal_HOG_SafePilot import predict_batches
from EXP_007_SVC_Normal_Full_Validation import features
from EXP_CMB_001_SVC_Combined_Hybrid_4Writer_OOF import atomic_json, write_csv
from EXP_CMB_003_KNN_Combined_Handmade_Augmentation import balanced_nested_indices, SEED, AUG_STRENGTH
from EXP007_KNN_Normal_Augmentation import size_jitter, stroke_jitter
from EXP011_KNN_Normal_Deskew import test_deskew
from EXP012_KNN_Normal_HOG import test_hog
from common import LABELS, fixed_split

CONTROL = ROOT / '05_model_optimize' / 'SVC' / 'Combined' / 'results' / 'EXP_CMB_001_SVC_Hybrid_4Writer_OOF'
OUT = ROOT / '05_model_optimize' / 'SVC' / 'Combined' / 'results' / 'EXP_CMB_003_SVC_Hybrid_Handmade_Augmentation'
EXPECTED_CONTROL = {41: 572, 42: 604, 43: 624, 44: 603}
FIELDS = ['ratio_percent', 'heldout_writer', 'train_writers', 'official_train_n', 'handmade_original_train_n',
          'handmade_augmented_train_n', 'train_n', 'handmade_correct', 'handmade_accuracy',
          'handmade_macro_f1', 'official_correct', 'official_accuracy', 'official_macro_f1',
          'fit_seconds', 'support_vectors', 'handmade_predict_seconds', 'official_predict_seconds']


def say(*items):
    print(*items, flush=True)


def read_control(handmade, sha):
    path = CONTROL / 'metrics.json'
    if not path.is_file():
        raise FileNotFoundError(f'Complete EXP-CMB-001 control is required: {path}')
    record = json.loads(path.read_text(encoding='utf-8'))
    if record.get('official_dataset_sha256') != sha or record.get('combined_oof_development', {}).get('correct') != 2403:
        raise RuntimeError('SVC Combined EXP-CMB-001 control mismatch')
    preds = {}
    for w in WRITERS:
        chk = record.get('handmade_sha256', {}).get(str(w))
        if chk != handmade[w][2]:
            raise RuntimeError(f'Control Hand-made SHA mismatch for writer {w}')
        p = CONTROL / f'fold_writer_{w}_predictions.csv'
        if not p.is_file():
            raise FileNotFoundError(f'Missing control fold predictions: {p}')
        with p.open(encoding='utf-8-sig', newline='') as f:
            rows = list(csv.DictReader(f))
        if len(rows) != 640 or any(int(row['index']) != i for i, row in enumerate(rows)):
            raise RuntimeError(f'Invalid control row order: writer {w}')
        if [LABELS.index(row['true']) for row in rows] != handmade[w][1].tolist():
            raise RuntimeError(f'Control truth mismatch: writer {w}')
        preds[w] = np.array([LABELS.index(row['combined_pred']) for row in rows], dtype=np.int64)
        if report(handmade[w][1], preds[w])['correct'] != EXPECTED_CONTROL[w]:
            raise RuntimeError(f'Control prediction mismatch: writer {w}')
    return record, preds


def load_checkpoint(path, csvpath, args, heldout, expected_train, official_sha, handmade, control_preds):
    if not path.exists() and not csvpath.exists():
        return None
    if path.exists() != csvpath.exists():
        raise RuntimeError(f'Incomplete checkpoint: {path} / {csvpath}')
    record = json.loads(path.read_text(encoding='utf-8'))
    if (record.get('status') != 'complete' or record.get('ratio_percent') != args.ratio
            or record.get('heldout_writer') != heldout or record.get('train_n') != expected_train
            or record.get('official_dataset_sha256') != official_sha
            or record.get('handmade_sha256') != {str(w): handmade[w][2] for w in WRITERS}
            or record.get('parameters') != {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf', 'cache_size_mb': args.cache_size}
            or record.get('augmentation_strength') != AUG_STRENGTH
            or record.get('augmentation_seed') != SEED + heldout):
        raise RuntimeError(f'Incompatible checkpoint: {path}')
    with csvpath.open(encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    if len(rows) != 640 or any(int(r['index']) != i for i, r in enumerate(rows)):
        raise RuntimeError(f'Invalid checkpoint predictions: {csvpath}')
    yt = handmade[heldout][1]
    if not np.array_equal(yt, [LABELS.index(r['true']) for r in rows]):
        raise RuntimeError('Checkpoint truth labels mismatch')
    if not np.array_equal(control_preds[heldout], [LABELS.index(r['control_pred']) for r in rows]):
        raise RuntimeError('Checkpoint control labels mismatch')
    pred = np.array([LABELS.index(r['augmented_pred']) for r in rows], dtype=np.int64)
    if report(yt, pred)['correct'] != record['handmade']['correct']:
        raise RuntimeError('Checkpoint scores mismatch')
    return record, pred


def run(args):
    if args.cache_size < 100:
        raise ValueError('--cache-size must be at least 100')
    test_deskew()
    test_hog()
    svc, scaled, source = baseline()
    if not scaled or svc.kernel != 'rbf':
        raise RuntimeError('Unexpected project baseline')
    say(f'SVC Combined EXP-CMB-003: augmentation ratio {args.ratio}%')
    say('FULL official Training=76800; writer training=1920 + augmented=', 1920*args.ratio//100)
    say('Official Test NOT loaded. Writer-heldout OOF is repeatedly used DEVELOPMENT data.')
    say('SVC source:', source)
    OUT.mkdir(parents=True, exist_ok=True)
    xtrain, ytrain, xval, yval, sha = fixed_split()
    if (len(ytrain), len(yval)) != (76800, 19200):
        raise RuntimeError('Official split sizes changed')
    handmade = {w: load_writer(w) for w in WRITERS}
    control, control_preds = read_control(handmade, sha)
    say('Control EXP-CMB-001 verified: 2403/2560, accuracy=0.938672')
    train_feat = features(xtrain, 'hybrid', 'Official Train 76800')
    del xtrain
    gc.collect()
    val_feat = features(xval, 'hybrid', 'Official Validation 19200')
    del xval
    gc.collect()
    feat_by_writer = {w: features(handmade[w][0], 'hybrid', f'Writer {w}') for w in WRITERS}
    if train_feat.shape != (76800, 1108) or val_feat.shape != (19200, 1108):
        raise RuntimeError('Unexpected feature shape')
    expected_aug = args.ratio * 1920 // 100
    expected_train = 78720 + expected_aug
    records = {}
    preds = {}
    for heldout in WRITERS:
        jsonpath = OUT / f'ratio_{args.ratio}_fold_writer_{heldout}.json'
        csvpath = OUT / f'ratio_{args.ratio}_fold_writer_{heldout}_predictions.csv'
        if not args.overwrite:
            found = load_checkpoint(jsonpath, csvpath, args, heldout, expected_train, sha, handmade, control_preds)
            if found is not None:
                records[heldout], preds[heldout] = found
                say(f'[Writer {heldout}] completed checkpoint reused')
                continue
        source_writers = [w for w in WRITERS if w != heldout]
        src_raw = np.concatenate([handmade[w][0] for w in source_writers])
        src_y = np.concatenate([handmade[w][1] for w in source_writers])
        src_writer = np.repeat(np.asarray(source_writers), 640)
        indices = balanced_nested_indices(src_writer, src_y, seed=SEED + heldout)[args.ratio]
        if len(indices) != expected_aug or np.any(src_writer[indices] == heldout):
            raise RuntimeError('Augmentation selection mismatch or heldout leak')
        say(f'\n[Fold heldout={heldout}] train writers={source_writers} augmented={len(indices)}')
        t = time.perf_counter()
        resized, size_stats = size_jitter(src_raw, indices, seed=SEED + heldout + 100)
        augmented_raw, stroke_stats = stroke_jitter(resized, strength=AUG_STRENGTH)
        del resized, src_raw
        aug_feat = features(augmented_raw, 'hybrid', f'Augmented fold {heldout}')
        del augmented_raw
        say(f'[Fold heldout={heldout}] augmentation prepared in {time.perf_counter()-t:.2f}s')
        src_feat = np.concatenate([feat_by_writer[w] for w in source_writers])
        train_x = np.concatenate([train_feat, src_feat, aug_feat])
        train_y = np.concatenate([ytrain, src_y, src_y[indices]])
        del src_feat, aug_feat, src_y
        gc.collect()
        if train_x.shape != (expected_train, 1108) or train_y.shape != (expected_train,):
            raise RuntimeError('Wrong train matrix shape')
        model = clone(svc).set_params(C=1.0, gamma='scale', cache_size=args.cache_size)
        say(f'[Fold heldout={heldout}] fit START rows={len(train_y)}')
        t = time.perf_counter()
        model.fit(train_x, train_y)
        fit_sec = round(time.perf_counter() - t, 2)
        del train_x, train_y
        gc.collect()
        say(f'[Fold heldout={heldout}] fit DONE {fit_sec}s support_vectors={len(model.support_)}')
        hp, hp_sec = predict_batches(model, feat_by_writer[heldout], f'Writer {heldout}', args.batch_size)
        hm = report(handmade[heldout][1], hp)
        say(f'[Fold heldout={heldout}] Hand-made accuracy={hm["accuracy"]:.6f} ({hm["correct"]}/640)')
        vp, vp_sec = predict_batches(model, val_feat, f'Official Val fold {heldout}', args.batch_size)
        vm = report(yval, vp)
        say(f'[Fold heldout={heldout}] Official Validation accuracy={vm["accuracy"]:.6f}')
        rec = {
            'status': 'complete', 'ratio_percent': args.ratio, 'heldout_writer': heldout,
            'train_writers': source_writers, 'official_dataset_sha256': sha,
            'handmade_sha256': {str(w): handmade[w][2] for w in WRITERS},
            'official_train_n': 76800, 'original_handmade_train_n': 1920,
            'augmented_handmade_train_n': expected_aug, 'train_n': expected_train,
            'augmentation_seed': SEED + heldout, 'augmentation_strength': AUG_STRENGTH,
            'size_stats': size_stats, 'stroke_stats': stroke_stats,
            'parameters': {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf', 'cache_size_mb': args.cache_size},
            'fit_seconds': fit_sec, 'support_vectors': int(len(model.support_)),
            'handmade_predict_seconds': hp_sec, 'official_predict_seconds': vp_sec,
            'handmade': hm, 'official_validation': vm,
        }
        pred_rows = [
            {'writer': heldout, 'index': i, 'true': LABELS[int(handmade[heldout][1][i])],
             'control_pred': LABELS[int(control_preds[heldout][i])],
             'augmented_pred': LABELS[int(hp[i])]} for i in range(640)
        ]
        write_csv(csvpath, pred_rows, ['writer', 'index', 'true', 'control_pred', 'augmented_pred'])
        atomic_json(jsonpath, rec)
        records[heldout] = rec
        preds[heldout] = hp
        del model, vp, hp
        gc.collect()
    yt = np.concatenate([handmade[w][1] for w in WRITERS])
    cp = np.concatenate([control_preds[w] for w in WRITERS])
    ap = np.concatenate([preds[w] for w in WRITERS])
    cm = confusion_matrix(yt, ap, labels=np.arange(16))
    write_csv(OUT / f'ratio_{args.ratio}_confusion_matrix.csv', [
        {'true': LABELS[i], **{LABELS[j]: int(cm[i,j]) for j in range(16)}} for i in range(16)
    ], ['true', *LABELS])
    paired = {
        'corrected': int(np.sum((cp != yt) & (ap == yt))),
        'new_errors': int(np.sum((cp == yt) & (ap != yt))),
        'unchanged_correct': int(np.sum((cp == yt) & (ap == yt))),
        'unchanged_wrong': int(np.sum((cp != yt) & (ap != yt))),
    }
    control_result = report(yt, cp)
    aug_result = report(yt, ap)
    fold_rows = []
    for w in WRITERS:
        rec = records[w]
        fold_rows.append({
            'ratio_percent': args.ratio, 'heldout_writer': w,
            'train_writers': ','.join(map(str, rec['train_writers'])),
            'official_train_n': 76800, 'handmade_original_train_n': 1920,
            'handmade_augmented_train_n': expected_aug, 'train_n': expected_train,
            'handmade_correct': rec['handmade']['correct'], 'handmade_accuracy': rec['handmade']['accuracy'],
            'handmade_macro_f1': rec['handmade']['macro_f1'],
            'official_correct': rec['official_validation']['correct'],
            'official_accuracy': rec['official_validation']['accuracy'],
            'official_macro_f1': rec['official_validation']['macro_f1'],
            'fit_seconds': rec['fit_seconds'], 'support_vectors': rec['support_vectors'],
            'handmade_predict_seconds': rec['handmade_predict_seconds'],
            'official_predict_seconds': rec['official_predict_seconds'],
        })
    write_csv(OUT / f'ratio_{args.ratio}_fold_results.csv', fold_rows, FIELDS)
    write_csv(OUT / f'ratio_{args.ratio}_oof_predictions.csv', [
        {'writer': w, 'index': i, 'true': LABELS[int(handmade[w][1][i])],
         'control_pred': LABELS[int(control_preds[w][i])], 'augmented_pred': LABELS[int(preds[w][i])]}
        for w in WRITERS for i in range(640)
    ], ['writer', 'index', 'true', 'control_pred', 'augmented_pred'])
    outcome = {
        'experiment': 'EXP-CMB-003 SVC Combined Hybrid train-only Hand-made augmentation',
        'ratio_percent': args.ratio, 'full_official_train_n': 76800, 'official_validation_n': 19200,
        'original_handmade_train_per_fold': 1920, 'augmented_handmade_train_per_fold': expected_aug,
        'train_per_fold': expected_train, 'official_dataset_sha256': sha,
        'feature': 'CoM -> Deskew -> hybrid pixel20/HOG80',
        'control_reference': str(CONTROL.relative_to(ROOT)),
        'control_oof': control_result, 'augmented_oof': aug_result,
        'change_percentage_points': 100 * (aug_result['accuracy'] - control_result['accuracy']),
        'paired': paired, 'folds': [records[w] for w in WRITERS],
        'official_test_used': False,
        'note': 'Hand-made OOF is repeatedly-used development data, NOT independent final Test.',
    }
    atomic_json(OUT / f'ratio_{args.ratio}_metrics.json', outcome)
    say(f'\nSVC COMBINED EXP-CMB-003 SUMMARY ratio={args.ratio}%')
    say(f'Control={control_result["accuracy"]:.6f} ({control_result["correct"]}/2560) F1={control_result["macro_f1"]:.6f}')
    say(f'Augmented={aug_result["accuracy"]:.6f} ({aug_result["correct"]}/2560) F1={aug_result["macro_f1"]:.6f}')
    say(f'Delta={outcome["change_percentage_points"]:+.4f}pp  Paired={paired}')
    say('Saved:', OUT)
    say('Official Test NOT loaded.')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--ratio', type=int, choices=[25, 50, 100], default=25)
    parser.add_argument('--batch-size', type=int, choices=[64, 128, 256, 512], default=256)
    parser.add_argument('--cache-size', type=int, default=200)
    parser.add_argument('--overwrite', action='store_true')
    run(parser.parse_args())


## EXP_CMB_004_SVC_Combined_Error_Analysis

원본: [05_model_optimize/SVC/Combined/scripts/EXP_CMB_004_SVC_Combined_Error_Analysis.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/scripts/EXP_CMB_004_SVC_Combined_Error_Analysis.py)  
Git blob: `2f3412dfc2234f7c2db570449451253a4cb7f310`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/logs)

Analyze saved SVC Combined writer-heldout predictions without retraining.



In [ ]:
%%archive_source 05_model_optimize/SVC/Combined/scripts/EXP_CMB_004_SVC_Combined_Error_Analysis.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Analyze saved SVC Combined writer-heldout predictions without retraining."""
from __future__ import annotations

import argparse
import csv
import json
from collections import Counter
from pathlib import Path

import numpy as np
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

ROOT = Path(__file__).resolve().parents[4]
BASE = ROOT / '05_model_optimize' / 'SVC' / 'Combined' / 'results'
AUG = BASE / 'EXP_CMB_003_SVC_Hybrid_Handmade_Augmentation'
OUT = BASE / 'EXP_CMB_004_SVC_Combined_Error_Analysis'
WRITERS = (41, 42, 43, 44)
RATIOS = (25, 50, 100)
EXPECTED = {0: 2403, 25: 2414, 50: 2424, 100: 2430}


def read_csv(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing experiment result: {path}')
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))


def save_csv(path, rows, fields):
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)


def load_predictions(ratio):
    path = AUG / f'ratio_{ratio}_oof_predictions.csv'
    rows = read_csv(path)
    result = {}
    for r in rows:
        key = (int(r['writer']), int(r['index']))
        if key in result:
            raise ValueError(f'Duplicate sample in ratio {ratio}: {key}')
        if key[0] not in WRITERS or not (0 <= key[1] < 640):
            raise ValueError(f'Unexpected writer/index: {key}')
        result[key] = {'true': r['true'], 'control': r['control_pred'], 'pred': r['augmented_pred']}
    expected_keys = {(w, i) for w in WRITERS for i in range(640)}
    if set(result) != expected_keys:
        raise ValueError(f'Ratio {ratio} has missing or unexpected samples')
    correct = sum(r['true'] == r['pred'] for r in result.values())
    control_correct = sum(r['true'] == r['control'] for r in result.values())
    if correct != EXPECTED[ratio] or control_correct != EXPECTED[0]:
        raise ValueError(f'Ratio {ratio} results changed: {correct} / control {control_correct}')
    return result


def analyze():
    data = {ratio: load_predictions(ratio) for ratio in RATIOS}
    reference = data[100]
    keys = sorted(reference)
    for ratio in RATIOS:
        for key in keys:
            if data[ratio][key]['true'] != reference[key]['true'] or data[ratio][key]['control'] != reference[key]['control']:
                raise ValueError(f'Inconsistent truth/control predictions for {key} at ratio {ratio}')
    labels = list('0123456789ABCDEF')
    if set(r['true'] for r in reference.values()) != set(labels):
        raise ValueError('Expected all 16 MNIST/EMNIST classes')
    for ratio in RATIOS:
        if not set(r['pred'] for r in data[ratio].values()) <= set(labels):
            raise ValueError(f'Unknown prediction label at ratio {ratio}')
    OUT.mkdir(parents=True, exist_ok=True)
    y = [reference[k]['true'] for k in keys]
    control = [reference[k]['control'] for k in keys]
    pred = [reference[k]['pred'] for k in keys]
    cm = confusion_matrix(y, pred, labels=labels)
    precision, recall, f1, support = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    per_class = []
    for i, lab in enumerate(labels):
        baseline_n = sum(reference[k]['true'] == lab and reference[k]['control'] == lab for k in keys)
        per_class.append({'class': lab, 'support': int(support[i]), 'correct': int(cm[i, i]),
                          'errors': int(support[i] - cm[i, i]), 'precision': float(precision[i]),
                          'recall': float(recall[i]), 'f1': float(f1[i]),
                          'control_correct': baseline_n, 'gain_vs_control': int(cm[i, i]) - baseline_n})
    save_csv(OUT / 'per_class.csv', per_class,
             ['class', 'support', 'correct', 'errors', 'precision', 'recall', 'f1', 'control_correct', 'gain_vs_control'])
    save_csv(OUT / 'confusion_matrix.csv',
             [{'true': lab, **{p: int(cm[i, j]) for j, p in enumerate(labels)}} for i, lab in enumerate(labels)],
             ['true', *labels])
    confusions = sorted(({'true': labels[i], 'predicted': labels[j], 'count': int(cm[i,j])}
                         for i in range(16) for j in range(16) if i != j and cm[i,j] > 0),
                        key=lambda z: (-z['count'], z['true'], z['predicted']))
    save_csv(OUT / 'top_confusions.csv', confusions, ['true','predicted','count'])
    writer_rows = []
    for w in WRITERS:
        samples = [k for k in keys if k[0] == w]
        cnt = sum(reference[k]['pred'] == reference[k]['true'] for k in samples)
        corrected = sum(reference[k]['control'] != reference[k]['true'] and reference[k]['pred'] == reference[k]['true'] for k in samples)
        regressed = sum(reference[k]['control'] == reference[k]['true'] and reference[k]['pred'] != reference[k]['true'] for k in samples)
        writer_rows.append({'writer': w, 'correct': cnt, 'incorrect': 640-cnt, 'accuracy': cnt/640,
                            'corrected_from_control': corrected, 'new_errors_vs_control': regressed})
    save_csv(OUT / 'per_writer.csv', writer_rows, ['writer','correct','incorrect','accuracy','corrected_from_control','new_errors_vs_control'])
    all_rows = []
    for w,i in keys:
        row = reference[(w,i)]
        status = ('corrected' if row['control'] != row['true'] and row['pred'] == row['true'] else
                  'new_error' if row['control'] == row['true'] and row['pred'] != row['true'] else
                  'still_wrong' if row['pred'] != row['true'] else 'still_correct')
        all_rows.append({'writer': w, 'index': i, 'true': row['true'], 'control_pred': row['control'],
                         'aug25_pred': data[25][(w,i)]['pred'], 'aug50_pred': data[50][(w,i)]['pred'],
                         'aug100_pred': row['pred'], 'control_correct': int(row['control'] == row['true']),
                         'aug100_correct': int(row['pred'] == row['true']), 'change_from_control': status})
    fields = list(all_rows[0])
    save_csv(OUT / 'all_samples.csv', all_rows, fields)
    save_csv(OUT / 'remaining_errors_100.csv', [r for r in all_rows if r['aug100_correct'] == 0], fields)
    save_csv(OUT / 'corrected_from_control.csv', [r for r in all_rows if r['change_from_control']=='corrected'], fields)
    save_csv(OUT / 'new_errors_vs_control.csv', [r for r in all_rows if r['change_from_control']=='new_error'], fields)
    ratio_rows = []
    for ratio in (0, *RATIOS):
        n = EXPECTED[ratio]
        ratio_rows.append({'augmentation_percent': ratio, 'correct': n, 'errors': 2560-n,
                           'accuracy': n/2560, 'delta_vs_control_pp': 100*(n-EXPECTED[0])/2560})
    save_csv(OUT / 'augmentation_comparison.csv', ratio_rows,
             ['augmentation_percent','correct','errors','accuracy','delta_vs_control_pp'])
    persistent = [r for r in all_rows if all(r[f'aug{x}_pred'] != r['true'] for x in RATIOS)]
    save_csv(OUT / 'errors_persistent_25_50_100.csv', persistent, fields)
    counts = Counter(r['change_from_control'] for r in all_rows)
    if len([r for r in all_rows if not r['aug100_correct']]) != 130 or counts['corrected'] != 28 or counts['new_error'] != 1:
        raise RuntimeError('100% augmentation summary does not match saved EXP-CMB-003 metrics')
    summary = {'experiment': 'EXP-CMB-004 SVC Combined saved prediction error analysis',
               'source': str(AUG.relative_to(ROOT)), 'official_test_used': False,
               'note': 'Repeated-use writer-heldout DEVELOPMENT samples, not independent final test',
               'n': 2560, 'correct': 2430, 'errors': 130, 'accuracy': 2430/2560,
               'paired_vs_control': dict(counts), 'persistent_errors_across_25_50_100': len(persistent),
               'top_confusions': confusions[:12], 'weakest_recall_classes': sorted(per_class,key=lambda r:r['recall'])[:8],
               'by_writer': writer_rows, 'augmentation': ratio_rows}
    (OUT / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nEXP-CMB-004: saved predictions only; NO SVC training / NO official Test', flush=True)
    for r in ratio_rows:
        print(f"Aug {r['augmentation_percent']:3d}%: {r['correct']}/2560 accuracy={r['accuracy']:.6f}",flush=True)
    print('100% augmented remaining errors:', 130,flush=True)
    print('Paired changes vs 0%:', dict(counts),flush=True)
    print('Top confusions:', [(r['true']+'->'+r['predicted'],r['count']) for r in confusions[:10]],flush=True)
    print('Weakest recalls:', [(r['class'],round(r['recall'],4)) for r in sorted(per_class,key=lambda r:r['recall'])[:6]],flush=True)
    print('Persistent errors across 25%, 50%, 100%:',len(persistent),flush=True)
    print('Saved:',OUT,flush=True)


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.parse_args()
    analyze()


## EXP_CMB_005_SVC_Combined_Targeted_DB_Augmentation

원본: [05_model_optimize/SVC/Combined/scripts/EXP_CMB_005_SVC_Combined_Targeted_DB_Augmentation.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/scripts/EXP_CMB_005_SVC_Combined_Targeted_DB_Augmentation.py)  
Git blob: `7e5396776c1a80de99af564542d035cb8cf0df9d`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Combined/logs)

SVC Combined EXP-CMB-005: 100% Hand-made augmentation plus targeted D/B augmentation.

Run from the repository root. Reuses saved EXP-CMB-003 ratio-100 predictions as control.
Official Training 76,800 in all folds, official Test never loaded.

CLI 인자 (정적 추출): `--extra-per-class-writer`, `--batch-size`, `--cache-size`, `--overwrite`

In [ ]:
%%archive_source 05_model_optimize/SVC/Combined/scripts/EXP_CMB_005_SVC_Combined_Targeted_DB_Augmentation.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SVC Combined EXP-CMB-005: 100% Hand-made augmentation plus targeted D/B augmentation.

Run from the repository root. Reuses saved EXP-CMB-003 ratio-100 predictions as control.
Official Training 76,800 in all folds, official Test never loaded.
"""
from __future__ import annotations

import argparse
import csv
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
from sklearn.base import clone
from sklearn.metrics import confusion_matrix

ROOT = Path(__file__).resolve().parents[4]
sys.path[:0] = [
    str(ROOT / '05_model_optimize' / 'SVC' / 'Combined' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'),
    str(ROOT / '05_model_optimize' / 'KNN' / 'Combined' / 'scripts'),
    str(ROOT / '03_experiments' / 'model_optimization'),
]
from EXP_001_SVC_Normal_Baseline import WRITERS, baseline, load_writer, report
from EXP_005_SVC_Normal_HOG_SafePilot import predict_batches
from EXP_007_SVC_Normal_Full_Validation import features
from EXP_CMB_001_SVC_Combined_Hybrid_4Writer_OOF import atomic_json, write_csv
from EXP_CMB_003_KNN_Combined_Handmade_Augmentation import balanced_nested_indices, SEED, AUG_STRENGTH
from EXP007_KNN_Normal_Augmentation import size_jitter, stroke_jitter
from EXP011_KNN_Normal_Deskew import test_deskew
from EXP012_KNN_Normal_HOG import test_hog
from common import LABELS, fixed_split

AUG100 = ROOT / '05_model_optimize' / 'SVC' / 'Combined' / 'results' / 'EXP_CMB_003_SVC_Hybrid_Handmade_Augmentation'
OUT = ROOT / '05_model_optimize' / 'SVC' / 'Combined' / 'results' / 'EXP_CMB_005_SVC_Targeted_DB_Augmentation'
EXPECTED_REFERENCE = {41: 580, 42: 610, 43: 629, 44: 611}
TARGET_CLASSES = (LABELS.index('D'), LABELS.index('B'))
TARGET_SEED = 20261127
FIELDS = ['extra_per_class_writer', 'heldout_writer', 'official_train_n', 'handmade_original_train_n',
          'handmade_base_augmented_n', 'handmade_targeted_augmented_n', 'train_n',
          'handmade_correct', 'handmade_accuracy', 'handmade_macro_f1', 'official_correct',
          'official_accuracy', 'official_macro_f1', 'fit_seconds', 'support_vectors']


def say(*args):
    print(*args, flush=True)


def read_reference(handmade, official_sha):
    path = AUG100 / 'ratio_100_metrics.json'
    if not path.is_file():
        raise FileNotFoundError(f'Missing completed EXP-CMB-003 100% control: {path}')
    result = json.loads(path.read_text(encoding='utf-8'))
    if (result.get('official_dataset_sha256') != official_sha or
            result.get('augmented_oof', {}).get('correct') != 2430 or
            result.get('ratio_percent') != 100):
        raise RuntimeError('EXP-CMB-003 100% control metrics mismatch')
    predictions = {}
    for w in WRITERS:
        p = AUG100 / f'ratio_100_fold_writer_{w}_predictions.csv'
        if not p.is_file():
            raise FileNotFoundError(f'Missing 100% control predictions: {p}')
        with p.open(encoding='utf-8-sig', newline='') as f:
            rows = list(csv.DictReader(f))
        if len(rows) != 640 or any(int(r['index']) != i for i, r in enumerate(rows)):
            raise RuntimeError(f'Control row order mismatch: writer {w}')
        truth = np.array([LABELS.index(r['true']) for r in rows], dtype=np.int64)
        pred = np.array([LABELS.index(r['augmented_pred']) for r in rows], dtype=np.int64)
        if not np.array_equal(truth, handmade[w][1]) or report(truth, pred)['correct'] != EXPECTED_REFERENCE[w]:
            raise RuntimeError(f'Control truth/predictions mismatch: writer {w}')
        predictions[w] = pred
    if sum(report(handmade[w][1], predictions[w])['correct'] for w in WRITERS) != 2430:
        raise RuntimeError('100% control overall incorrect')
    return predictions


def targeted_indices(writer_ids, labels, per_group, seed):
    rng = np.random.default_rng(seed)
    groups = []
    for w in sorted(set(int(w) for w in writer_ids)):
        for cls in TARGET_CLASSES:
            candidates = np.flatnonzero((writer_ids == w) & (labels == cls))
            if len(candidates) != 40:
                raise RuntimeError(f'Expected 40 source examples for writer={w} class={LABELS[cls]}')
            groups.append(rng.permutation(candidates)[:per_group])
    selected = np.sort(np.concatenate(groups))
    if len(selected) != 3 * 2 * per_group or not np.isin(labels[selected], TARGET_CLASSES).all():
        raise RuntimeError('Targeted augmentation selection invalid')
    return selected


def checkpoint(jsonpath, csvpath, args, w, sha, handmade, control_preds, expected_n):
    if not jsonpath.exists() and not csvpath.exists():
        return None
    if jsonpath.exists() != csvpath.exists():
        raise RuntimeError(f'Incomplete checkpoint: {jsonpath}, {csvpath}')
    rec = json.loads(jsonpath.read_text(encoding='utf-8'))
    expected_hashes = {str(k): handmade[k][2] for k in WRITERS}
    if (rec.get('status') != 'complete' or rec.get('heldout_writer') != w or
            rec.get('extra_per_class_writer') != args.extra_per_class_writer or
            rec.get('train_n') != expected_n or rec.get('official_dataset_sha256') != sha or
            rec.get('handmade_sha256') != expected_hashes or
            rec.get('target_seed') != TARGET_SEED or
            rec.get('augmentation_seed') != SEED + w or
            rec.get('augmentation_strength') != AUG_STRENGTH or
            rec.get('parameters') != {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf',
                                       'cache_size_mb': args.cache_size}):
        raise RuntimeError(f'Incompatible checkpoint: {jsonpath}')
    with csvpath.open(encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    if len(rows) != 640 or any(int(r['index']) != i for i, r in enumerate(rows)):
        raise RuntimeError('Invalid checkpoint row order')
    true = np.array([LABELS.index(r['true']) for r in rows], dtype=np.int64)
    control = np.array([LABELS.index(r['control_pred']) for r in rows], dtype=np.int64)
    pred = np.array([LABELS.index(r['targeted_pred']) for r in rows], dtype=np.int64)
    if (not np.array_equal(true, handmade[w][1]) or
            not np.array_equal(control, control_preds[w]) or
            report(true, pred)['correct'] != rec['handmade']['correct']):
        raise RuntimeError('Checkpoint prediction mismatch')
    return rec, pred


def run(args):
    if args.cache_size < 100:
        raise ValueError('--cache-size must be >= 100')
    test_deskew()
    test_hog()
    svc, scaled, source = baseline()
    if not scaled or svc.kernel != 'rbf':
        raise RuntimeError('Unexpected repository SVC baseline')
    n_target = 6 * args.extra_per_class_writer
    n_total = 76800 + 1920 + 1920 + n_target
    prefix = f'extra_{args.extra_per_class_writer}'
    say('EXP-CMB-005: FULL official Training=76800; 4-writer heldout; 100% base augmentation')
    say(f'Targeted D/B: {args.extra_per_class_writer} per writer/class, +{n_target} per fold; total rows={n_total}')
    say('SVC C=1, gamma=scale; CoM -> Deskew -> Pixel:HOG 20:80')
    say('Official Test NOT loaded; Hand-made is repeatedly-used DEVELOPMENT data.')
    say('SVC source:', source)
    xtr, ytr, xv, yv, sha = fixed_split()
    if (len(ytr), len(yv)) != (76800, 19200):
        raise RuntimeError('Official data split changed')
    handmade = {w: load_writer(w) for w in WRITERS}
    control_preds = read_reference(handmade, sha)
    say('100% augmented control verified: 2430/2560 (94.9219%)')
    OUT.mkdir(parents=True, exist_ok=True)
    official_feat = features(xtr, 'hybrid', 'Official Train 76800')
    del xtr
    gc.collect()
    validation_feat = features(xv, 'hybrid', 'Official Validation 19200')
    del xv
    gc.collect()
    writer_feat = {w: features(handmade[w][0], 'hybrid', f'Writer {w}') for w in WRITERS}
    if official_feat.shape != (76800, 1108) or validation_feat.shape != (19200, 1108):
        raise RuntimeError('Unexpected feature shapes')
    records, predictions = {}, {}
    for heldout in WRITERS:
        jsonpath = OUT / f'{prefix}_fold_writer_{heldout}.json'
        csvpath = OUT / f'{prefix}_fold_writer_{heldout}_predictions.csv'
        if not args.overwrite:
            saved = checkpoint(jsonpath, csvpath, args, heldout, sha, handmade,
                               control_preds, n_total)
            if saved is not None:
                records[heldout], predictions[heldout] = saved
                say(f'[Fold heldout={heldout}] complete checkpoint reused')
                continue
        train_writers = [w for w in WRITERS if w != heldout]
        raw = np.concatenate([handmade[w][0] for w in train_writers])
        source_y = np.concatenate([handmade[w][1] for w in train_writers])
        writer_ids = np.repeat(np.array(train_writers, dtype=np.int64), 640)
        if set(writer_ids) != set(train_writers) or heldout in writer_ids:
            raise RuntimeError('Heldout writer leakage')
        all_base_indices = balanced_nested_indices(writer_ids, source_y, seed=SEED+heldout)[100]
        if len(all_base_indices) != 1920:
            raise RuntimeError('Base augmentation index mismatch')
        targets = targeted_indices(writer_ids, source_y,
                                   args.extra_per_class_writer, TARGET_SEED+heldout)
        say(f'\n[Fold heldout={heldout}] writers={train_writers}; base aug=1920; D/B extra={len(targets)}')
        t = time.perf_counter()
        resized_base, base_size_stats = size_jitter(raw, all_base_indices,
                                                    seed=SEED+heldout+100)
        base_raw, base_stroke_stats = stroke_jitter(resized_base, strength=AUG_STRENGTH)
        del resized_base
        base_feat = features(base_raw, 'hybrid', f'Base augmentation fold {heldout}')
        del base_raw
        resized_target, target_size_stats = size_jitter(raw, targets,
                                                        seed=TARGET_SEED+heldout+1000)
        target_raw, target_stroke_stats = stroke_jitter(resized_target, strength=AUG_STRENGTH)
        del resized_target, raw
        target_feat = features(target_raw, 'hybrid', f'D/B augmentation fold {heldout}')
        del target_raw
        say(f'[Fold heldout={heldout}] augmentation prepared {time.perf_counter()-t:.2f}s')
        original_feat = np.concatenate([writer_feat[w] for w in train_writers])
        fit_x = np.concatenate([official_feat, original_feat, base_feat, target_feat])
        fit_y = np.concatenate([ytr, source_y, source_y[all_base_indices], source_y[targets]])
        del original_feat, base_feat, target_feat, source_y
        gc.collect()
        if fit_x.shape != (n_total, 1108) or fit_y.shape != (n_total,):
            raise RuntimeError('Combined training shapes invalid')
        model = clone(svc).set_params(C=1.0, gamma='scale', cache_size=args.cache_size)
        say(f'[Fold heldout={heldout}] fit START rows={n_total}')
        t = time.perf_counter()
        model.fit(fit_x, fit_y)
        fit_sec = round(time.perf_counter()-t, 2)
        del fit_x, fit_y
        gc.collect()
        say(f'[Fold heldout={heldout}] fit DONE {fit_sec}s support_vectors={len(model.support_)}')
        pred, hp_sec = predict_batches(model, writer_feat[heldout],
                                       f'Writer {heldout}', args.batch_size)
        hm = report(handmade[heldout][1], pred)
        say(f'[Fold heldout={heldout}] Hand-made={hm["accuracy"]:.6f} ({hm["correct"]}/640)')
        val_pred, val_sec = predict_batches(model, validation_feat,
                                             f'Official Val fold {heldout}', args.batch_size)
        vm = report(yv, val_pred)
        say(f'[Fold heldout={heldout}] Official Validation={vm["accuracy"]:.6f}')
        rec = {
            'status': 'complete', 'heldout_writer': heldout,
            'train_writers': train_writers, 'extra_per_class_writer': args.extra_per_class_writer,
            'official_dataset_sha256': sha,
            'handmade_sha256': {str(w): handmade[w][2] for w in WRITERS},
            'official_train_n': 76800, 'original_handmade_train_n': 1920,
            'base_augmented_handmade_n': 1920, 'targeted_augmented_handmade_n': n_target,
            'train_n': n_total, 'augmentation_seed': SEED+heldout,
            'augmentation_strength': AUG_STRENGTH, 'target_seed': TARGET_SEED,
            'base_size_stats': base_size_stats, 'base_stroke_stats': base_stroke_stats,
            'target_size_stats': target_size_stats, 'target_stroke_stats': target_stroke_stats,
            'parameters': {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf',
                           'cache_size_mb': args.cache_size},
            'fit_seconds': fit_sec, 'support_vectors': int(len(model.support_)),
            'handmade_predict_seconds': hp_sec, 'official_predict_seconds': val_sec,
            'handmade': hm, 'official_validation': vm,
        }
        write_csv(csvpath, [
            {'writer': heldout, 'index': i, 'true': LABELS[int(handmade[heldout][1][i])],
             'control_pred': LABELS[int(control_preds[heldout][i])],
             'targeted_pred': LABELS[int(pred[i])]} for i in range(640)
        ], ['writer', 'index', 'true', 'control_pred', 'targeted_pred'])
        atomic_json(jsonpath, rec)
        records[heldout], predictions[heldout] = rec, pred
        del model, val_pred, pred
        gc.collect()
    truth = np.concatenate([handmade[w][1] for w in WRITERS])
    control = np.concatenate([control_preds[w] for w in WRITERS])
    candidate = np.concatenate([predictions[w] for w in WRITERS])
    baseline_result = report(truth, control)
    result = report(truth, candidate)
    paired = {
        'corrected': int(np.sum((control != truth) & (candidate == truth))),
        'new_errors': int(np.sum((control == truth) & (candidate != truth))),
        'unchanged_correct': int(np.sum((control == truth) & (candidate == truth))),
        'unchanged_wrong': int(np.sum((control != truth) & (candidate != truth))),
    }
    cm = confusion_matrix(truth, candidate, labels=np.arange(16))
    write_csv(OUT / f'{prefix}_confusion_matrix.csv', [
        {'true': LABELS[i], **{LABELS[j]: int(cm[i,j]) for j in range(16)}} for i in range(16)
    ], ['true', *LABELS])
    write_csv(OUT / f'{prefix}_oof_predictions.csv', [
        {'writer': w, 'index': i, 'true': LABELS[int(handmade[w][1][i])],
         'control_pred': LABELS[int(control_preds[w][i])],
         'targeted_pred': LABELS[int(predictions[w][i])]} for w in WRITERS for i in range(640)
    ], ['writer', 'index', 'true', 'control_pred', 'targeted_pred'])
    fold_rows = []
    for w in WRITERS:
        rec = records[w]
        fold_rows.append({
            'extra_per_class_writer': args.extra_per_class_writer, 'heldout_writer': w,
            'official_train_n': 76800, 'handmade_original_train_n': 1920,
            'handmade_base_augmented_n': 1920, 'handmade_targeted_augmented_n': n_target,
            'train_n': n_total, 'handmade_correct': rec['handmade']['correct'],
            'handmade_accuracy': rec['handmade']['accuracy'],
            'handmade_macro_f1': rec['handmade']['macro_f1'],
            'official_correct': rec['official_validation']['correct'],
            'official_accuracy': rec['official_validation']['accuracy'],
            'official_macro_f1': rec['official_validation']['macro_f1'],
            'fit_seconds': rec['fit_seconds'], 'support_vectors': rec['support_vectors'],
        })
    write_csv(OUT / f'{prefix}_fold_results.csv', fold_rows, FIELDS)
    classes = []
    for name in ('D', 'B', '0', '8'):
        label = LABELS.index(name)
        mask = truth == label
        classes.append({
            'class': name, 'n': int(mask.sum()),
            'control_correct': int((control[mask] == label).sum()),
            'targeted_correct': int((candidate[mask] == label).sum()),
            'control_recall': float(np.mean(control[mask] == label)),
            'targeted_recall': float(np.mean(candidate[mask] == label)),
        })
    write_csv(OUT / f'{prefix}_critical_classes.csv', classes,
              ['class', 'n', 'control_correct', 'targeted_correct', 'control_recall', 'targeted_recall'])
    output = {
        'experiment': 'EXP-CMB-005 SVC Combined targeted D/B after 100% augmentation',
        'extra_per_class_writer': args.extra_per_class_writer,
        'official_training_n': 76800, 'official_validation_n': 19200,
        'original_handmade_n_per_fold': 1920, 'base_augmented_n_per_fold': 1920,
        'targeted_augmented_n_per_fold': n_target, 'train_n_per_fold': n_total,
        'official_dataset_sha256': sha, 'official_test_used': False,
        'feature': 'CoM -> Deskew -> Pixel20/HOG80',
        'reference_experiment': str(AUG100.relative_to(ROOT)),
        'reference_oof': baseline_result, 'targeted_oof': result,
        'change_percentage_points': 100 * (result['accuracy'] - baseline_result['accuracy']),
        'paired': paired, 'critical_classes': classes,
        'folds': [records[w] for w in WRITERS],
        'note': 'Hand-made OOF repeatedly used for model development, not independent Test.',
    }
    atomic_json(OUT / f'{prefix}_metrics.json', output)
    say(f'\nEXP-CMB-005 SUMMARY D/B +{n_target} /fold')
    say(f'100% control={baseline_result["accuracy"]:.6f} ({baseline_result["correct"]}/2560)')
    say(f'Targeted    ={result["accuracy"]:.6f} ({result["correct"]}/2560), MacroF1={result["macro_f1"]:.6f}')
    say(f'Change={output["change_percentage_points"]:+.4f}pp  Paired={paired}')
    say('Critical class recalls:', [(r['class'], round(r['control_recall'],4),
                                      round(r['targeted_recall'],4)) for r in classes])
    say('Official Validation by fold:', [round(records[w]['official_validation']['accuracy'],6)
                                          for w in WRITERS])
    say('Saved:', OUT)
    say('Official Test NOT loaded.')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--extra-per-class-writer', type=int, choices=[20, 40], default=20)
    parser.add_argument('--batch-size', type=int, choices=[64, 128, 256, 512], default=256)
    parser.add_argument('--cache-size', type=int, default=200)
    parser.add_argument('--overwrite', action='store_true')
    run(parser.parse_args())


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")